In [1]:
import pandas as pd
from pandas_plink import read_plink1_bin, write_plink1_bin
import numpy as np
import anndata
import os

/cis/home/xhan56/anaconda3/envs/mmt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adni_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_1_GWAS_Plink/ADNI_cluster_01_forward_757LONI'
adni_2_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_GO_2_OmniExpress/ADNI_GO_2_Forward_Bin'
adni_2_2_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI_GO2_2nd/Binary/ADNI_GO2_GWAS_2nd_orig_BIN'
adni_3_1_path = '/export/io79/data/schaud35/datasets/adni/genomic/ADNI3_PLINKFinal/PLINK_Final/ADNI3_PLINK_Final'
adni_3_2_path = '/export/io79/data/schaud35/datasets/adni/genomic/PLINKFinal_ADNI_20231107/PLINK_Final/ADNI3_PLINK_FINAL_2nd'
FILTERED_BIM_SUFFIX = '_allele_filtered'  # filtered SNPs written here; original .bim files are not modified
GENOMIC_OUT_DIR = '/export/io79/data/schaud35/datasets/adni_processed/genomic'  # all notebook-generated outputs

# 1. Prepare liftover format

In [3]:
adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
os.makedirs(GENOMIC_OUT_DIR, exist_ok=True)

for i in range(5):
    df = pd.read_csv(adni_paths[i] + '.bim', header=None, sep='\t')

    valid_bases = ['A', 'C', 'G', 'T']
    df_filtered = df[df[4].isin(valid_bases) & df[5].isin(valid_bases)]
    df_filtered = df_filtered.reset_index(drop=True)
    out_bim = os.path.join(GENOMIC_OUT_DIR, os.path.basename(adni_paths[i]) + FILTERED_BIM_SUFFIX + '.bim')
    df_filtered.to_csv(out_bim, index=False, header=None, sep='\t')

In [4]:
adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
save_paths = ['adni_1.bed', 'adni_2_1.bed', 'adni_2_2.bed', 'adni_3_1.bed', 'adni_3_2.bed']
before_liftover_dir = os.path.join(GENOMIC_OUT_DIR, 'before_liftover')
os.makedirs(before_liftover_dir, exist_ok=True)

for i in range(len(adni_paths)):
    bim_path = adni_paths[i]
    save_path = os.path.join(before_liftover_dir, save_paths[i])
    filtered_bim = os.path.join(GENOMIC_OUT_DIR, os.path.basename(bim_path) + FILTERED_BIM_SUFFIX + '.bim')
    tmp = pd.read_csv(filtered_bim, header=None, sep='\t')
    
    chr_id = tmp[0].replace({23: 'X', 24: 'Y'}).where((0 < tmp[0]) & (tmp[0] < 25), None)
    
    filtered_data = tmp[chr_id.notna()]
    
    new_tmp = pd.DataFrame({
        '0': 'chr' + chr_id[chr_id.notna()].astype(str),
        '1': filtered_data[3],
        '2': filtered_data[3],
        '3': filtered_data[1]
    })
    
    new_tmp = new_tmp.reset_index(drop=True)
    new_tmp.to_csv(save_path, index=False, sep='\t', header=None)

# 2. Run Liftover & Filtering

In [ ]:
### --------------- Liftover Process (Run following commandas at the terminal) --------------- ###
# wget http://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/liftOver
# chmod +x liftOver
# wget http://hgdownload.soe.ucsc.edu/goldenPath/hg18/liftOver/hg18ToHg38.over.chain.gz
# liftOver /export/io79/data/schaud35/datasets/adni_processed/genomic/before_liftover/adni_1.bed hg18ToHg38.over.chain.gz /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/ADNI_1_Hg38.bed /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/unlifted_ADNI_1_Hg38.bed
# liftOver /export/io79/data/schaud35/datasets/adni_processed/genomic/before_liftover/adni_2_1.bed hg18ToHg38.over.chain.gz /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/ADNI_2_1_Hg38.bed /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/unlifted_ADNI_2_1_Hg38.bed
# liftOver /export/io79/data/schaud35/datasets/adni_processed/genomic/before_liftover/adni_2_2.bed hg18ToHg38.over.chain.gz /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/ADNI_2_2_Hg38.bed /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/unlifted_ADNI_2_2_Hg38.bed
# liftOver /export/io79/data/schaud35/datasets/adni_processed/genomic/before_liftover/adni_3_1.bed hg18ToHg38.over.chain.gz /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/ADNI_3_1_Hg38.bed /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/unlifted_ADNI_3_1_Hg38.bed
# liftOver /export/io79/data/schaud35/datasets/adni_processed/genomic/before_liftover/adni_3_2.bed hg18ToHg38.over.chain.gz /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/ADNI_3_2_Hg38.bed /export/io79/data/schaud35/datasets/adni_processed/genomic/after_liftover/unlifted_ADNI_3_2_Hg38.bed

In [5]:
# filtering
after_liftover_dir = os.path.join(GENOMIC_OUT_DIR, 'after_liftover')
os.makedirs(after_liftover_dir, exist_ok=True)
after_liftover_paths = ['after_liftover/ADNI_1', 'after_liftover/ADNI_2_1', 'after_liftover/ADNI_2_2', 'after_liftover/ADNI_3_1', 'after_liftover/ADNI_3_2']

for after_liftover_path in after_liftover_paths:
    bed_file = os.path.join(GENOMIC_OUT_DIR, after_liftover_path + '_Hg38.bed')
    _tmp = pd.read_csv(bed_file, header=None, sep='\t')
    _tmp_filtered = _tmp[~_tmp[0].str.contains('_')]
    _tmp_filtered.to_csv(bed_file, index=False, header=False, sep='\t')

# 3. Align .bim, .bed, .fam with liftover ids

In [6]:
adni_paths = [adni_1_path, adni_2_1_path, adni_2_2_path, adni_3_1_path, adni_3_2_path]
liftover_paths = ['after_liftover/ADNI_1', 'after_liftover/ADNI_2_1', 'after_liftover/ADNI_2_2', 'after_liftover/ADNI_3_1', 'after_liftover/ADNI_3_2']
output_paths = ['liftovered/ADNI_1', 'liftovered/ADNI_2_1', 'liftovered/ADNI_2_2', 'liftovered/ADNI_3_1', 'liftovered/ADNI_3_2']
liftovered_dir = os.path.join(GENOMIC_OUT_DIR, 'liftovered')
os.makedirs(liftovered_dir, exist_ok=True)

for i in range(5):
    adni_path = adni_paths[i]
    liftover_path = os.path.join(GENOMIC_OUT_DIR, liftover_paths[i] + '_Hg38.bed')
    output_path = os.path.join(GENOMIC_OUT_DIR, output_paths[i])
    filtered_bim = os.path.join(GENOMIC_OUT_DIR, os.path.basename(adni_paths[i]) + FILTERED_BIM_SUFFIX + '.bim')

    G = read_plink1_bin(adni_path + '.bed', filtered_bim, adni_path + '.fam')

    # Read the liftover file to get the new SNP IDs
    liftover = pd.read_csv(liftover_path, sep='\t', header=None, names=['chr', 'pos1', 'pos2', 'id'])

    # Find indices of the SNPs in bim that are present in the liftover file
    indices = np.arange(G.shape[1])[(pd.DataFrame(G.snp)[0].isin(liftover['id']))]

    if i == 0:
        fam = pd.read_csv(adni_paths[0] + '.fam', sep=' ', header=None, names=['fid', 'iid', 'father', 'mother', 'gender', 'trait'])
        fam_indices = ~fam['iid'].isin(['073_S_0909', '130_S_1201'])
        _G = G[fam_indices, indices]

    else:
        _G = G[:, indices]

    _G['chrom'] = ('variant', np.array(liftover['chr']))
    _G['pos'] = ('variant', np.array(liftover['pos1']))
    write_plink1_bin(_G, output_path + '.bed', output_path + '.bim', output_path + '.fam')

Mapping files: 100%|██████████| 3/3 [00:00<00:00,  5.02it/s]

Writing BED: 100%|██████████| 2/2 [00:09<00:00,  4.73s/it]

Writing FAM... done.
Writing BIM... 

done.


Writing BED: 100%|██████████| 2/2 [00:07<00:00,  3.99s/it]

...

Writing FAM... done.
Writing BIM... 

done.


Writing BED: 100%|██████████| 1/1 [00:05<00:00,  5.06s/it]

Writing FAM... done.
Writing BIM... 

done.


In [7]:
G_list = []
for i in range(5):
    adni_path = os.path.join(GENOMIC_OUT_DIR, output_paths[i])

    G = read_plink1_bin(adni_path + '.bed', adni_path + '.bim', adni_path + '.fam')
    G_list.append(G)

Mapping files: 100%|██████████| 3/3 [00:00<00:00,  3.27it/s]


# 4.Merge using plink on your local enviroment

dont run below cell echo and plink commands, keep reading

In [ ]:
### --------------- Merge using plink (run the following commands in a terminal) --------------- ###
# No sudo: install PLINK 1.9 under your home directory (run from the folder where you want plink_install/).
# wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20241022.zip
# unzip plink_linux_x86_64_20241022.zip -d plink_install
# mkdir -p "$HOME/bin"
# cp plink_install/plink "$HOME/bin/plink" && chmod +x "$HOME/bin/plink"
# export PATH="$HOME/bin:$PATH"   # this shell only; add the same line to ~/.bashrc for future logins (no sudo)
# Alternative: skip PATH and call the binary by full path, e.g. "$HOME/.../plink_install/plink" instead of plink below.
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1" > /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets.txt
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1 --merge-list /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets.txt --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged

run following cell instead of above:

note -- Pipeline initially failed because cohort PLINK merge produced multiallelic/strand conflicts (Error: variants with 3+ alleles present) after liftover, so ADNI_final.bed was never created and downstream read_plink1_bin() errored (No BED file has been found).
We resolved this by applying PLINK harmonization (--flip on missnp candidates, then --exclude ADNI_merged-merge.missnp) before re-merging; this enables successful output generation but may reduce final SNP count (variant-level information loss) while generally preserving sample count.

In [ ]:
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1      --exclude /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged-merge.missnp --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1_excl
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1_flip --exclude /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged-merge.missnp --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1_flip_excl
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2_flip --exclude /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged-merge.missnp --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2_flip_excl
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1_flip --exclude /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged-merge.missnp --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1_flip_excl
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2_flip --exclude /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged-merge.missnp --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2_flip_excl

# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1_flip_excl" > /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets_flip_excl.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2_flip_excl" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets_flip_excl.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1_flip_excl" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets_flip_excl.txt
# echo "/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2_flip_excl" >> /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets_flip_excl.txt

# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1_excl \
#   --merge-list /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/all_datasets_flip_excl.txt \
#   --make-bed \
#   --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged

In [ ]:
### --------------- LD pruning using plink (Run following commandas at the terminal) --------------- ###
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged --indep-pairwise 50 5 0.1 --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged_pruned
# plink --bfile /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged --extract /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_merged_pruned.prune.in --make-bed --out /export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_final

In [8]:
merged_path = os.path.join(GENOMIC_OUT_DIR, 'liftovered', 'ADNI_final')
G_merge = read_plink1_bin(merged_path + '.bed', merged_path + '.bim', merged_path + '.fam')

genotype_df = G_merge.to_pandas()
genotype_df.columns = G_merge.snp

adata = anndata.AnnData(X=genotype_df.values,
                        obs=pd.DataFrame(index=genotype_df.index),
                        var=pd.DataFrame(index=genotype_df.columns))

adata.write_h5ad(os.path.join(GENOMIC_OUT_DIR, 'genomic_merged.h5ad'))

Mapping files: 100%|██████████| 3/3 [00:00<00:00,  3.90it/s]


In [9]:
import os, glob
print("GENOMIC_OUT_DIR:", GENOMIC_OUT_DIR)
print("ADNI_final exists?", os.path.exists(os.path.join(GENOMIC_OUT_DIR, "liftovered", "ADNI_final.bed")))
print("Available liftovered BEDs:", glob.glob(os.path.join(GENOMIC_OUT_DIR, "liftovered", "*.bed")))

GENOMIC_OUT_DIR: /export/io79/data/schaud35/datasets/adni_processed/genomic
ADNI_final exists? True
Available liftovered BEDs: ['/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_1_flip.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_2_2_flip.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_1_flip.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_3_2_flip.bed', '/export/io79/data/schaud35/datasets/adni_processed/genomic/liftovered/ADNI_1_ex